# Example: Diagnosing an infeasible model with Gurobi's IIS

This example builds a deliberately infeasible single-region energy system and diagnoses it
with Gurobi's **Irreducible Inconsistent Subsystem (IIS)**.

When a model cannot be solved, the solver usually reports nothing but `infeasible`. That gives
no hint about which part of the model is impossible, so the common reaction is trial and error:
switching components off one by one until the model solves again. On a large energy system
model this is slow and error-prone.

Gurobi can do better. For an infeasible model it computes an IIS: a minimal set of constraints
and variable bounds that cannot be satisfied together, and from which removing any single
member makes the rest satisfiable. The IIS usually points straight at the modeling mistake,
for example a fixed demand that no source can meet.

The workflow is structured as follows:
1. Import required packages
2. Create an infeasible energy system model
3. Optimize the model
4. Compute the IIS with Gurobi
5. Repair the model
6. Other solvers

## 1. Import required packages

In [1]:
from pathlib import Path

import gurobipy as gp
import pandas as pd

import fine as fn

%load_ext autoreload
%autoreload 2

## 2. Create an infeasible energy system model

The model has one region, one commodity and four hourly time steps. The only source is a PV
component whose capacity is capped at 1 GW, so it supplies at most 1 GWh per hour. The sink
fixes the electricity demand at 10 GW in every hour. The commodity balance can therefore not
be closed in any time step: the model is infeasible by construction, which makes the diagnosis
below reproducible.

In [2]:
locations = {"Region"}
commodityUnitDict = {"electricity": r"GW$_{el}$"}
commodities = {"electricity"}
numberOfTimeSteps = 4
hoursPerTimeStep = 1

In [3]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=numberOfTimeSteps,
    commodityUnitsDict=commodityUnitDict,
    hoursPerTimeStep=hoursPerTimeStep,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
)

### PV with a capacity cap of 1 GW

In [4]:
esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=pd.Series(1.0, index=list(locations)),
    )
)

### Fixed demand of 10 GW that the source cannot meet

In [5]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=pd.DataFrame(
            10.0, index=range(numberOfTimeSteps), columns=list(locations)
        ),
    )
)

## 3. Optimize the model

The model is infeasible by construction, so the run ends without a solution. This is all the
solver reports on its own: the problem is named, its cause is not.

In [6]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0028 sec)

Declaring shared potential constraint...
		(0.0001 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.0001 sec)

		(0.0000 sec)

Declaring objective function...
		(0.0026 sec)

Either solver not selected or specified solver not available.gurobi is set as solver.
Read LP format model from file /tmp/tmpp8pt98xu.pyomo.lp
Reading time = 0.00 seconds
x1: 11 rows, 13 columns, 22 nonzeros
Set parameter QCPDual to value 1
Set parameter Threads to value 3
Set parameter LogFile to value ""
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD Ryzen 9 7950X 16-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a 

## 4. Compute the IIS with Gurobi

`esM.pyM` is the Pyomo model behind the energy system. It exists as soon as the optimization
problem has been declared, which `esM.optimize()` does before handing the problem to the
solver. On a large model the failed solve can be skipped by calling
`esM.declareOptimizationProblem()` instead.

The Pyomo model is written to an LP file and read into Gurobi. Use
`symbolic_solver_labels=True`: without it, Pyomo names the constraints `c1, c2, ...` and the
IIS cannot be traced back to the model.

In [7]:
lp_file = Path("infeasible_model.lp")
esM.pyM.write(str(lp_file), io_options={"symbolic_solver_labels": True})

('infeasible_model.lp', 135394493278352)

In [8]:
gurobi_model = gp.read(str(lp_file))
gurobi_model.setParam("OutputFlag", 0)  # silence Gurobi's console log

# computeIIS() returns nothing. Instead it marks every constraint and bound that belongs
# to the IIS via the attributes IISConstr, IISLB and IISUB.
gurobi_model.computeIIS()

# Write only the IIS to a file: same LP format, reduced to the conflicting part.
iis_file = Path("infeasible_model.ilp")
gurobi_model.write(str(iis_file))

Read LP format model from file infeasible_model.lp
Reading time = 0.00 seconds
Obj: 11 rows, 13 columns, 22 nonzeros
Reading time = 0.00 seconds
Obj: 11 rows, 13 columns, 22 nonzeros


In [9]:
conflicting_constraints = [
    constraint.ConstrName
    for constraint in gurobi_model.getConstrs()
    if constraint.IISConstr
]
print(f"{len(conflicting_constraints)} constraint(s) in the IIS:")
for constraint_name in conflicting_constraints:
    print(f"  {constraint_name}")

print("\nBound(s) in the IIS:")
for variable in gurobi_model.getVars():
    if variable.IISLB:
        print(f"  {variable.VarName} >= {variable.LB}")
    if variable.IISUB:
        print(f"  {variable.VarName} <= {variable.UB}")

2 constraint(s) in the IIS:
  c_u_ConstrOperation1_srcSnk(Region_PV_0_0_0)_
  c_e_commodityBalanceConstraint(Region_electricity_0_0_0)_

Bound(s) in the IIS:
  cap_srcSnk(Region_PV_0) <= 1.0
  op_srcSnk(Region_Electricity_demand_0_0_0) >= 10.0


### 4.1 Read the `.ilp` file

In [10]:
print(iis_file.read_text())

\ Model Obj_copy
\ LP format - for model browsing. Use MPS format to capture full model detail.
\ Signature: 0x64d302b18a6c5bbf
Minimize
 
Subject To
 c_u_ConstrOperation1_srcSnk(Region_PV_0_0_0)_: - cap_srcSnk(Region_PV_0)
   + op_srcSnk(Region_PV_0_0_0) <= 0
 c_e_commodityBalanceConstraint(Region_electricity_0_0_0)_:
   op_srcSnk(Region_PV_0_0_0) - op_srcSnk(Region_Electricity_demand_0_0_0)
   = 0
Bounds
 -infinity <= cap_srcSnk(Region_PV_0) <= 1
 op_srcSnk(Region_PV_0_0_0) free
 op_srcSnk(Region_Electricity_demand_0_0_0) >= 10
End



The LP format splits a model into two parts:

- **`Bounds`**: limits on a single variable, such as `cap_PV <= 1` (PV capacity is at most
  1 GW) and `op_demand >= 10` (demand is at least 10 GW). The fixed numbers of the model
  live here.
- **`Subject To`**: constraints that link several variables:
  - `op_PV = op_demand`: the commodity balance; PV must generate exactly what is consumed.
  - `op_PV <= cap_PV`: PV cannot generate more than its capacity.
.

Read together, the two parts spell out the contradiction: demand is 10, the balance forces PV
to generate 10, but PV can generate at most 1. The bounds hold the numbers you would change;
the constraints are the rules that make them collide.

**Note:** The IIS contains *one* minimal conflict, not all of them. If a model has several
independent errors, the IIS shows one of them. After repairing it, the model can still be
infeasible and the next IIS reveals the next conflict: diagnosis is iterative.

## 5. Repair the model

The IIS contains `cap_PV <= 1`, which collides with the fixed demand of 10 GW. Raising exactly
this bound, `capacityMax` from 1 GW to 10 GW, resolves the contradiction and leaves the rest
of the model unchanged.

In [11]:
esM.removeComponent("PV")

esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityMax=pd.Series(10.0, index=list(locations)),
    )
)

In [12]:
esM.optimize()
esM.getOptimizationSummary("SourceSinkModel")

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0014 sec)

Declaring shared potential constraint...
		(0.0000 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.0001 sec)

		(0.0000 sec)

Declaring objective function...
		(0.0024 sec)

Either solver not selected or specified solver not available.gurobi is set as solver.
Read LP format model from file /tmp/tmppqa5f_in.pyomo.lp
Reading time = 0.00 seconds
x1: 11 rows, 13 columns, 22 nonzeros
Set parameter QCPDual to value 1
Set parameter Threads to value 3
Set parameter LogFile to value ""
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD Ryzen 9 7950X 16-Core Processor, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 16 physical cores, 32 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a 

Region
Component          Property                        Unit                    
Electricity demand NPVcontribution                 [1e9 Euro]           0.0
                   TAC                             [1e9 Euro/a]         0.0
                   capacity                        [GW$_{el}$]          NaN
                   capexCap                        [1e9 Euro/a]         NaN
                   capexIfBuilt                    [1e9 Euro/a]         NaN
                   commissioning                   [GW$_{el}$]          NaN
                   commodCosts                     [1e9 Euro/a]         0.0
                   commodRevenues                  [1e9 Euro/a]         0.0
                   decommissioning                 [GW$_{el}$]          NaN
                   invest                          [1e9 Euro]           NaN
                   investLifetimeExtension         [1e9 Euro]           NaN
                   isBuilt                         [-]                  NaN
                   operation                       [GW$_{el}$*h/a]  87600.0
                                                   [GW$_{el}$*h]       40.0
                   opexCap                         [1e9 Euro/a]         NaN
                   opexIfBuilt                     [1e9 Euro/a]         NaN
                   opexOp                          [1e9 Euro/a]         0.0
                   revenueLifetimeShorteningResale [1e9 Euro]           NaN
PV                 NPVcontribution                 [1e9 Euro]           0.0
                   TAC                             [1e9 Euro/a]         0.0
                   capacity                        [GW$_{el}$]         10.0
                   capexCap                        [1e9 Euro/a]         0.0
                   capexIfBuilt                    [1e9 Euro/a]         NaN
                   commissioning                   [GW$_{el}$]         10.0
                   commodCosts                     [1e9 Euro/a]         0.0
                   commodRevenues                  [1e9 Euro/a]         0.0
                   decommissioning                 [GW$_{el}$]          0.0
                   invest                          [1e9 Euro]           0.0
                   investLifetimeExtension         [1e9 Euro]             0
                   isBuilt                         [-]                  NaN
                   operation                       [GW$_{el}$*h/a]  87600.0
                                                   [GW$_{el}$*h]       40.0
                   opexCap                         [1e9 Euro/a]         0.0
                   opexIfBuilt                     [1e9 Euro/a]         NaN
                   opexOp                          [1e9 Euro/a]         0.0
                   revenueLifetimeShorteningResale [1e9 Euro]             0

## 6. Other solvers

IIS computation is a Gurobi feature. Among the open-source solvers,
[HiGHS](https://highs.dev/) offers a similar but more limited function,
[`getIis()`](https://ergo-code.github.io/HiGHS/dev/guide/advanced/#highs-iis).